# Search-11d — Descente sous budget : la loi derrière les métaheuristiques

**Série** : Search / Part1-Foundations — variante -d de [Search-11](Search-11-Metaheuristics.ipynb) (métaheuristiques MEALPy) et [Search-11c](Search-11c-Empirical-Algorithm-Selection.ipynb) (sélection empirique)
**Opération étudiée** : **op 11 « Descendre sous budget »**, après `mimo_lean/Descent.lean` (Lean-formel)
**Source primaire (lue firsthand)** : `MyIA.AI.Notebooks/SymbolicAI/Lean/mimo_lean/Descent.lean` — `descent_flips_le_barrier` (confinement + plafond) et `descent_target_before_ceiling` (terminaison dans la cible avant épuisement du budget), thèse op 11 explicite l.148

**Ce que ce notebook fait que Search-11 ne fait pas** : Search-11 *utilise* des métaheuristiques qui améliorent iterativement sous un budget d'évaluations ; il n'énonce ni ne mesure jamais la **loi** qui garantit qu'elles terminent. Ici, la loi est un objet : énoncée, **exercée par des assertions runtime sur des centaines d'exécutions**, et surtout **cassée où elle doit casser** — sa troisième hypothèse est fausse en paysage générique, et le notebook l'exhibe.

**Statuts de preuve** : *[exercée ici]* = mesuré/asserted dans ce notebook · *[Lean-formel]* = prouvé dans `Descent.lean` · *[rapporté]* = affirmé sans re-vérification.

Exécution : stdlib uniquement (`random`, `itertools`), CPU, ~20 min de lecture, ~5 s d'exécution.


**Navigation** : [Search-11c](Search-11c-Empirical-Algorithm-Selection.ipynb) | [Version C#](Search-11d-Descente-Sous-Budget-CSharp.ipynb) | [Search-12](Search-12-BranchAndBound-Borne-LP-Python.ipynb)

## 1. La loi, et ce que l'attestation Lean prouve exactement

`Descent.lean` formalise une descente comme une suite d'états où seuls les flips **acceptés** comptent, sous trois hypothèses :

| Hypothèse | Énoncé | Rôle |
|---|---|---|
| `hstrict` | chaque mouvement accepté décroît **strictement** le coût | interdit les cycles |
| `hbarrier` | le coût reste **confiné** sous une borne `B` sur tout le run | majore le nombre de pas |
| `hnostall` | hors de la cible, un mouvement accepté existe **toujours** | bloque uniquement sur la cible |

Sous ces trois hypothèses, `descent_target_before_ceiling` démontre : le dernier état est **dans la cible**, et le run a effectué **strictement moins de flips acceptés que le plafond** `M_N` dès que `B < M_N`. C'est la garantie de complexité : borne de flips pour un run terminal ; ce plafond n’est pas un budget d’évaluations.

La question de ce notebook : **que reste-t-il de cette loi quand on quitte le cadre Lean** — c'est-à-dire sur un paysage combinatoire générique, avec un budget d'évaluations explicite ? Réponse en trois temps : la loi de terminaison **tient** (§2, exercée), la troisième hypothèse **tombe** (§3, exhibée), et la qualité est mesurée selon le budget (§4).


In [1]:
import random
import itertools
import statistics
import sys

random.seed(20260916)
print(f"Python {sys.version.split()[0]}, stdlib only")

Python 3.13.16, stdlib only


## 2. L'objet : une descente sous budget, instrumentée

Le substrat : **sac à dos 0/1** (instances synthétiques seedées). Le potentiel Φ = valeur négative de la sélection (décroître Φ = augmenter la valeur). Le voisinage : flips de bit uniques. La barrière de confinement : la **relaxation LP** du sac à dos (majorant de toute sélection admissible — Φ ne peut pas descendre en dessous).

Chaque exécution est **instrumentée** : trajectoire de Φ, mouvements acceptés, raison d'arrêt. Les assertions `hstrict`/`hbarrier` sont vérifiées **au runtime, à chaque mouvement accepté** — la loi n'est pas décrite, elle est exercée.

Pour relier le modèle au coût naturel de Lean, on peut prendre `floor(lp) - valeur(x)` : les valeurs sont entières et admissibles. Le nombre de flips acceptés est alors borné par ce coût initial, tandis que les évaluations comptent aussi les voisins rejetés. Un arrêt sur budget ne certifie pas la terminalité exigée par Lean.

In [2]:
def instance_knapsack(n, seed, ratio_poids=0.5):
    """Instance 0/1 synthetique : n objets, capacite = ratio * somme poids."""
    rng = random.Random(seed)
    poids = [rng.randint(10, 100) for _ in range(n)]
    valeurs = [poids[i] + rng.randint(-5, 20) for i in range(n)]  # correlation valeur/poids
    capacite = int(ratio_poids * sum(poids))
    return poids, valeurs, capacite

def borne_lp(poids, valeurs, capacite):
    """Relaxation fractionnaire (greedy densite) : majorant de l'optimum 0/1."""
    ordre = sorted(range(len(poids)), key=lambda i: valeurs[i] / poids[i], reverse=True)
    reste, total = capacite, 0
    for i in ordre:
        pris = min(poids[i], reste)
        total += valeurs[i] * pris / poids[i]
        reste -= pris
        if reste == 0:
            break
    return total

def dp_exact(poids, valeurs, capacite):
    """Optimum 0/1 exact par programmation dynamique (temoin de verite)."""
    best = [0] * (capacite + 1)
    for p, v in zip(poids, valeurs):
        for c in range(capacite, p - 1, -1):
            if best[c - p] + v > best[c]:
                best[c] = best[c - p] + v
    return best[capacite]

def admissible(x, poids, capacite):
    return sum(p for p, xi in zip(poids, x) if xi) <= capacite

def valeur(x, valeurs):
    return sum(v for v, xi in zip(valeurs, x) if xi)

def descente_sous_budget(x0, poids, valeurs, capacite, budget, collect=True):
    """Hill-climbing a stricte amelioration, plafonne a `budget` evaluations.

    Retourne un dict : etat final, trajectoire du potentiel, nombre de pas,
    raison d'arret ('budget' | 'blocage'), et compteurs de violation des
    hypotheses hstrict/hbarrier (assertions runtime de la loi, section 2).
    """
    lp = borne_lp(poids, valeurs, capacite)
    x = list(x0)
    phi = -valeur(x, valeurs)          # potentiel : decroitre = s'ameliorer
    trajectoire = [phi] if collect else []
    hstrict_viol, hbarrier_viol = 0, 0
    evaluations, pas = 0, 0
    ameliorant_final = None
    while evaluations < budget:
        best_delta, best_i = 0, None
        explores = 0
        for i in range(len(x)):
            if evaluations >= budget:
                break
            explores += 1
            y = list(x); y[i] = 1 - y[i]
            if not admissible(y, poids, capacite):
                evaluations += 1
                continue
            evaluations += 1
            d = valeur(y, valeurs) - valeur(x, valeurs)
            if d > best_delta:
                best_delta, best_i = d, i
        if best_i is None:
            # Un balayage incomplet ne prouve pas l'absence d'ameliorant.
            ameliorant_final = False if explores == len(x) else None
            break
        y = list(x); y[best_i] = 1 - y[best_i]
        phi_new = -valeur(y, valeurs)
        if phi_new >= phi:             # hstrict : mouvement accepté doit decroitre strictement
            hstrict_viol += 1
        if -phi_new > lp + 1e-9:       # hbarrier : confinement sous le majorant LP
            hbarrier_viol += 1
        x, phi = y, phi_new
        pas += 1
        if collect:
            trajectoire.append(phi)
    else:
        ameliorant_final = None        # budget epuise avant balayage complet du voisinage
    if ameliorant_final is None:
        # le dernier balayage incomplet peut cacher un ameliorant : le noter honnetement
        raison = 'budget'
    else:
        raison = 'blocage'
    return {'x': x, 'valeur': -phi, 'trajectoire': trajectoire, 'pas': pas,
            'evaluations': evaluations, 'raison': raison,
            'hstrict_viol': hstrict_viol, 'hbarrier_viol': hbarrier_viol,
            'lp': lp}

print("Fonctions sac à dos initialisées : LP, DP exacte, admissibilité, valeur et descente instrumentée.")


Fonctions sac à dos initialisées : LP, DP exacte, admissibilité, valeur et descente instrumentée.


### Lecture de l'instrumentation

- `raison` distingue **budget atteint** (le plafond d'évaluations a interrompu la recherche alors qu'un améliorant pouvait exister) de **blocage** (un balayage **complet** du voisinage n'a trouvé aucun mouvement améliorant) — la dissociation que `Descent.lean` érige en objet.
- `hstrict_viol` / `hbarrier_viol` comptent les violations des deux premières hypothèses **sur les mouvements réellement acceptés** : ils doivent rester à zéro, et la cellule suivante le vérifie en série.
- `lp` est le majorant LP : la barrière de confinement. Aucun état admissible ne peut dépasser cette valeur — Φ reste confiné au-dessus de −lp.

In [3]:
# LA LOI, EXERCICEE : hstrict + hbarrier + plafond, sur 60 executions
random.seed(20260916)
runs = []
for n in (16, 24, 32):
    for seed in range(20):
        poids, valeurs, capacite = instance_knapsack(n, seed)
        x0 = [0] * n
        budget = 8 * n * n            # generosite : budget >> longueur typique d'un run
        r = descente_sous_budget(x0, poids, valeurs, capacite, budget)
        runs.append((n, seed, r))

v_strict = sum(r['hstrict_viol'] for _, _, r in runs)
v_barrier = sum(r['hbarrier_viol'] for _, _, r in runs)
depassements = sum(1 for (n, _, r) in runs if r['evaluations'] > 8 * n * n)

print(f"executions              : {len(runs)}")
print(f"violations hstrict      : {v_strict}   (mouvements acceptes sans decroissance stricte)")
print(f"violations hbarrier     : {v_barrier}   (etat au-dela du majorant LP)")
print(f"depassements de plafond : {depassements}   (evaluations > budget)")
assert v_strict == 0 and v_barrier == 0 and depassements == 0
for _, _, r in runs:
    assert len(r['trajectoire']) == r['pas'] + 1
    assert all(b < a for a, b in zip(r['trajectoire'], r['trajectoire'][1:]))
    assert r['pas'] <= int(r['lp']) + r['trajectoire'][0]
    assert all(-phi <= r['lp'] + 1e-9 for phi in r['trajectoire'])
print("=> *[exercee ici]* la loi de terminaison tient : strictement decroissant, confine, plafonne.")
# Un balayage partiel n'est pas un certificat de blocage.
a = descente_sous_budget([0, 0], [2, 1], [1, 2], 1, 1)
b = descente_sous_budget([0, 0], [2, 1], [1, 2], 1, 0)
c = descente_sous_budget([0, 1], [2, 1], [1, 2], 1, 2)
assert (a['raison'], b['raison'], c['raison']) == ('budget', 'budget', 'blocage')
print('fixtures partielle, zero et balayage complet : OK')

# Depart non nul : borne de flips calculee depuis la valeur initiale.
t = descente_sous_budget([1, 0], [1, 1], [1, 4], 2, 10)
t_sans = descente_sous_budget([1, 0], [1, 1], [1, 4], 2, 10, collect=False)
assert t['trajectoire'] == [-1, -5] and t['pas'] == 1
assert t['pas'] <= int(t['lp']) - 1 and t_sans['trajectoire'] == []
print("trajectoire, depart non nul et collect=False : OK")


executions              : 60
violations hstrict      : 0   (mouvements acceptes sans decroissance stricte)
violations hbarrier     : 0   (etat au-dela du majorant LP)
depassements de plafond : 0   (evaluations > budget)
=> *[exercee ici]* la loi de terminaison tient : strictement decroissant, confine, plafonne.
fixtures partielle, zero et balayage complet : OK
trajectoire, depart non nul et collect=False : OK


### Interprétation

Les compteurs ci-dessus contrôlent les mouvements effectivement acceptés et le budget d'évaluations sur ces 60 runs. La borne naturelle `floor(lp) - valeur(x0)` majore les flips acceptés ; elle ne majore pas directement le nombre d'évaluations. La terminaison sous budget vient ici du compteur explicite, et un arrêt sur budget ne prouve pas la terminalité de Lean.

La décroissance stricte du potentiel et son confinement restent des contrôles distincts : le premier exclut les cycles de mouvements acceptés, le second borne leur nombre. Un balayage peut cependant essayer plusieurs voisins avant d'accepter un flip ; le budget d'évaluations se consomme aussi sur les essais rejetés. Le transport de la borne Lean porte donc sur les flips et sur ses hypothèses, tandis que l'arrêt effectif de cette implémentation est assuré par son budget explicite.


### Exercice 1 — redémarrages stratégiques

La section suivante étudiera les blocages locaux. Répartir un budget global entre `k_restarts` descentes, en demandant chaque point initial à `x0_gen` et en appelant le callback `descendre(x, budget_partiel)` pour chacun. Retourner le meilleur objet obtenu de ce callback, sans le modifier. Le harnais compte les appels et budgets réellement alloués, y compris budget nul et capacité nulle. Une fois le stub complété, il mesure les 60 instances de la section 3, avec et sans quatre redémarrages, au même budget global de 2304 évaluations. Aucun gain n'est supposé.

**Méthode.** Garder les mêmes poids, valeurs et capacité pour les comparaisons : seuls les points de départ et l'allocation du budget changent. Conserver chaque objet retourné par la descente et choisir celui dont la valeur est maximale ; les évaluations déjà payées lors d'un redémarrage restent imputées au budget global. Le contrôle observe les callbacks pour distinguer une véritable série de descentes d'une valeur construite sans exécution.

**Lecture attendue.** Comparer les valeurs et les raisons d'arrêt, puis expliquer si le partage du budget permet de quitter certains bassins locaux. Plusieurs essais plus courts peuvent explorer davantage de bassins, mais aussi manquer de temps pour progresser dans chacun : une amélioration sur toutes les graines n'est pas une conséquence du redémarrage. Le cas de budget nul doit produire un résultat cohérent sans dépenser d'évaluation.


In [4]:
def descente_restart(x0_gen, poids, valeurs, capacite, budget_global, k_restarts, descendre):
    # TODO : appeler x0_gen et descendre pour chacun des k departs ;
    # repartir le budget entre les appels, puis retourner le meilleur resultat obtenu.
    return None

def verifier_restarts(departs, poids, valeurs, capacite, budget, k):
    fournis, resultats, allocations = [], [], []
    def prochain():
        x = list(departs())
        assert admissible(x, poids, capacite)
        fournis.append(x[:])
        return x
    def descendre(x, allocation):
        assert len(resultats) < len(fournis) and list(x) == fournis[len(resultats)]
        assert isinstance(allocation, int) and allocation >= 0
        allocations.append(allocation)
        assert sum(allocations) <= budget
        r = descente_sous_budget(x, poids, valeurs, capacite, allocation)
        resultats.append((r, list(r['x']), r['valeur'], r['evaluations']))
        return r
    r = descente_restart(prochain, poids, valeurs, capacite, budget, k, descendre)
    if r is None:
        return None
    assert len(fournis) == len(resultats) == k and sum(allocations) <= budget
    assert sum(e for _, _, _, e in resultats) <= budget
    assert any(r is sortie and r['x'] == x and r['valeur'] == v and r['evaluations'] == e
               for sortie, x, v, e in resultats)
    assert r['valeur'] == max(v for _, _, v, _ in resultats)
    assert len(r['x']) == len(poids) and all(b in (0, 1) for b in r['x'])
    assert admissible(r['x'], poids, capacite) and valeur(r['x'], valeurs) == r['valeur']
    return r

def sequence_departs(sequence):
    it = iter(sequence)
    return lambda: next(it)

r1 = verifier_restarts(sequence_departs([[0, 0], [0, 1]]), [2, 1], [1, 2], 1, 20, 2)
r2 = verifier_restarts(sequence_departs([[0], [0]]), [1], [7], 0, 10, 2)
r0 = verifier_restarts(sequence_departs([[0, 0], [0, 1]]), [2, 1], [1, 2], 1, 0, 2)
if r1 is None or r2 is None or r0 is None:
    print("Exercice 1 a completer")
else:
    assert (r1['valeur'], r2['valeur'], r0['valeur']) == (2, 0, 2)
    assert r0['evaluations'] == 0
    print("Exercice 1 : appels, budgets et fixtures valides")
    # Meme protocole et budget que la section 3, 60 instances ; aucun gain suppose.
    avec, sans = [], []
    categories_sans = {"budget": 0, "cible": 0, "hors cible": 0}
    categories_avec = {"budget": 0, "cible": 0, "hors cible": 0}
    for seed in range(60):
        p, v, cap = instance_knapsack(24, seed)
        rng = random.Random(seed)
        def depart():
            x = [rng.randint(0, 1) for _ in p]
            return x if admissible(x, p, cap) else [0] * len(p)
        initial = depart()
        base = descente_sous_budget(initial, p, v, cap, 2304, collect=False)
        rng = random.Random(seed)
        r = verifier_restarts(depart, p, v, cap, 2304, 4)
        opt = dp_exact(p, v, cap)
        for resultat, categories in ((base, categories_sans), (r, categories_avec)):
            categorie = ('budget' if resultat['raison'] == 'budget' else
                         'cible' if resultat['valeur'] == opt else 'hors cible')
            categories[categorie] += 1
        sans.append(base['valeur'] / opt)
        avec.append(r['valeur'] / opt)
    print(f"60 instances, budget 2304 : ratio median sans restart {statistics.median(sans):.1%}, avec {statistics.median(avec):.1%}")
    print(f"arrets sans restart {categories_sans} ; meilleur restart {categories_avec}")


Exercice 1 a completer


## 3. Le contre-claim : `hnostall` ne survit pas au paysage générique

Le théorème Lean conclut « le dernier état est **dans la cible** » — mais seulement sous `hnostall` : *hors de la cible, un mouvement accepté existe toujours*. Cette hypothèse est gratuite en paysage générique. Ce qui suit l'**exhibe** : on classe chaque arrêt en **budget atteint** vs **blocage**, puis on demande aux blocages s'ils sont sur la cible (optimum global, connu par DP exacte) ou **hors cible** — des optima locaux, la négation exacte de `hnostall`.

In [5]:
# LA DISSOCIATION, EXERCICEE : budget-atteint vs blocage, et blocage-hors-cible
random.seed(20260916)
budgets_serre = 4 * 24 * 24           # budget volontairement serre pour rendre 'budget' observable
stats = {'budget': 0, 'blocage_cible': 0, 'blocage_hors_cible': 0}
gaps_hors_cible = []
n = 24
for seed in range(60):
    poids, valeurs, capacite = instance_knapsack(n, seed)
    opt = dp_exact(poids, valeurs, capacite)
    rng = random.Random(seed)
    x0 = [rng.randint(0, 1) for _ in range(n)]
    if not admissible(x0, poids, capacite):
        x0 = [0] * n
    r = descente_sous_budget(x0, poids, valeurs, capacite, budgets_serre, collect=False)
    if r['raison'] == 'budget':
        stats['budget'] += 1
        gap = 1 - r['valeur'] / opt
        gaps_hors_cible.append(gap)   # budget epuise : gap quelconque
    else:
        if r['valeur'] == opt:
            stats['blocage_cible'] += 1
        else:
            stats['blocage_hors_cible'] += 1
            gaps_hors_cible.append(1 - r['valeur'] / opt)

total = sum(stats.values())
print(f"runs                        : {total}")
print(f"budget atteint              : {stats['budget']:3d}  ({stats['budget']/total:.0%})")
print(f"blocage SUR la cible        : {stats['blocage_cible']:3d}  ({stats['blocage_cible']/total:.0%})")
print(f"blocage HORS cible (optimum local) : {stats['blocage_hors_cible']:3d}  ({stats['blocage_hors_cible']/total:.0%})")
print(f"gap final median            : {statistics.median(gaps_hors_cible):.1%}")
print(f"gap final max               : {max(gaps_hors_cible):.1%}")
print()
print("=> *[exercee ici]* hnostall est fausse :", end=" ")
if stats['blocage_hors_cible'] > 0:
    print(f"{stats['blocage_hors_cible']} runs se bloquent sur un optimum local :")
    print("   le theoreme 'dernier etat dans la cible' ne se transporte PAS ;")
    print("   sa conclusion devient : dernier etat = minimum LOCAL, atteint sous budget.")
else:
    print("aucun optimum local observe sur cet echantillon (a confronter a un voisinage plus pauvre).")

# Regime budget SERRE (~2 balayages) : la cause d'arret 'budget' existe aussi
stats_tight = {'budget': 0, 'blocage': 0}
for seed in range(30):
    poids, valeurs, capacite = instance_knapsack(n, seed)
    rng = random.Random(seed)
    x0 = [rng.randint(0, 1) for _ in range(n)]
    if not admissible(x0, poids, capacite):
        x0 = [0] * n
    r = descente_sous_budget(x0, poids, valeurs, capacite, 48, collect=False)
    stats_tight[r['raison']] += 1
print(f"a budget serre (48 evals ~ 2 balayages) : budget {stats_tight['budget']}/30, blocage {stats_tight['blocage']}/30")

runs                        : 60
budget atteint              :   0  (0%)
blocage SUR la cible        :   0  (0%)
blocage HORS cible (optimum local) :  60  (100%)
gap final median            : 10.0%
gap final max               : 16.0%

=> *[exercee ici]* hnostall est fausse : 60 runs se bloquent sur un optimum local :
   le theoreme 'dernier etat dans la cible' ne se transporte PAS ;
   sa conclusion devient : dernier etat = minimum LOCAL, atteint sous budget.


a budget serre (48 evals ~ 2 balayages) : budget 25/30, blocage 5/30


### Interprétation — le défaut comme dissociation

Les comptages ci-dessus distinguent les balayages complets sans amélioration des interruptions sur budget. Un blocage hors cible est un témoin contre `hnostall` pour ce voisinage ; un arrêt sur budget ne permet pas de conclure à un optimum local. Les points de départ sont tirés avec un générateur initialisé une seule fois par graine.

Dans cette exécution Python, 60 blocages sur 60 sont hors cible (gap médian 10,0 %) ; à 48 évaluations, 25 arrêts sur budget et 5 blocages sont observés.

La distinction est celle de deux informations différentes. Après un balayage complet sans améliorant, le voisinage inspecté certifie seulement un optimum local ; la DP exacte permet alors de constater s'il est également global. Après une interruption sur budget, les voisins non examinés peuvent encore contenir un améliorant : ni blocage ni qualité globale ne sont établis. La réserve `hnostall` est ainsi une hypothèse substantielle sur le paysage et le voisinage, et non un effet que l'on peut déduire de la seule terminaison du programme.


### Exercice 2 — barrière : LP fractionnaire vs borne triviale somme des valeurs

**Contexte.** La section 2 vérifie `hbarrier` à chaque mouvement accepté : aucun état ne doit dépasser le majorant LP `lp` calculé par `borne_lp`. La loi de terminaison borne le nombre de pas par la barrière : une barrière plus serrée donne un plafond plus petit.

**Objectif.** Écrire `plafond_theorique(valeur_initiale, borne)` : le majorant du nombre de pas acceptés, à partir de la valeur initiale de `x0` et de la barrière `lp`. Comparer ensuite ce majorant à celui obtenu avec la borne triviale somme des valeurs, sur les mêmes instances.

**Indices.**
- `hbarrier` est une hypothèse : une barrière plus serrée raccourcit le plafond théorique.
- Recalculer le majorant de flips de `descent_flips_le_barrier` (`Descent.lean`) avec `lp`, contre la valeur initiale.
- Comparer au pas réellement observé (champ `pas` renvoyé par `descente_sous_budget`).

**Critère de vérification.** Le majorant est un nombre supérieur ou égal au champ `pas` de chaque run de la section 2 ; un majorant inférieur au pas observé signale une erreur de barrière.

In [6]:
def plafond_theorique(valeur_initiale, borne):
    # TODO : retourner le plafond de flips pour une borne entiere
    return None

r1, r2 = plafond_theorique(3, 10), plafond_theorique(7, 9)
if r1 is None or r2 is None:
    print("Exercice 2 a completer")
else:
    assert (r1, r2) == (7, 2)
    print("Exercice 2 : fixtures valides")

Exercice 2 a completer


## 4. Ce que le budget achète : courbe qualité-budget

La descente est un algorithme *anytime* : à chaque plafond correspond une qualité atteignable. La mesure : gap final vs optimum DP exact, à budget croissant, 30 graines par point — la courbe de rendement du budget.

In [7]:
# COURBE QUALITE-BUDGET (anytime) : gap median vs budget, multi-seed
random.seed(20260916)
n = 24
echantillons = []
for B in (20, 50, 100, 200, 400, 800):
    gaps = []
    for seed in range(30):
        poids, valeurs, capacite = instance_knapsack(n, 1000 + seed)
        opt = dp_exact(poids, valeurs, capacite)
        rng = random.Random(2000 + seed)
        x0 = [rng.randint(0, 1) for _ in range(n)]
        if not admissible(x0, poids, capacite):
            x0 = [0] * n
        r = descente_sous_budget(x0, poids, valeurs, capacite, B, collect=False)
        gaps.append(1 - r['valeur'] / opt)
    echantillons.append((B, gaps))
    q = sorted(gaps)
    print(f"budget {B:4d} evaluations | gap median {statistics.median(gaps):6.1%} | "
          f"IQR [{q[7]:.1%}, {q[22]:.1%}] | pire {max(gaps):.1%}")

g_med = [statistics.median(g) for _, g in echantillons]
ameliorations = [g_med[i] - g_med[i + 1] for i in range(len(g_med) - 1)]
print()
print("rendement marginal (gain de gap par doubling de budget) :", )
for i, a in enumerate(ameliorations):
    print(f"  {echantillons[i][0]:4d} -> {echantillons[i+1][0]:4d} : {a:+.1%}")

budget   20 evaluations | gap median  86.3% | IQR [18.4%, 87.1%] | pire 89.0%
budget   50 evaluations | gap median  63.8% | IQR [12.2%, 66.4%] | pire 72.6%


budget  100 evaluations | gap median  39.3% | IQR [12.2%, 42.8%] | pire 48.0%


budget  200 evaluations | gap median  10.8% | IQR [7.9%, 12.7%] | pire 17.0%
budget  400 evaluations | gap median   8.9% | IQR [7.0%, 12.2%] | pire 17.0%


budget  800 evaluations | gap median   8.9% | IQR [7.0%, 12.2%] | pire 17.0%

rendement marginal (gain de gap par doubling de budget) :
    20 ->   50 : +22.4%
    50 ->  100 : +24.5%
   100 ->  200 : +28.5%
   200 ->  400 : +1.9%
   400 ->  800 : +0.0%


### Interprétation

Les statistiques ci-dessus décrivent les 30 instances et ces budgets ; elles ne constituent pas une garantie générale de rendement décroissant. Un plateau peut provenir de blocages locaux ; il faut examiner les raisons d'arrêt pour l'établir. Changer de voisinage ou redémarrer peut modifier la qualité à budget égal.

Le gap médian mesuré passe de 86,3 % à 20 évaluations à 8,9 % à 400 puis 800 évaluations.

### Exercice 3 — voisinage un ou deux flips

Étendre la descente aux flips simples **et** aux paires, sous le même budget. L'union contient le voisinage à un flip ; un optimum local à un flip peut rester local pour le voisinage élargi, ou cesser de l'être. Pour chaque candidat, appeler `evaluer(depuis, voisin)`, qui compte effectivement l'évaluation et retourne la valeur ou `None` si le voisin est inadmissible. Retourner un dictionnaire avec `x`, `valeur`, `chemin` (liste des états acceptés, départ inclus), `evaluations` et `raison` (`budget` ou `blocage`).

Le contrôle vérifie états et valeurs recalculées, trace des mouvements évalués, budget nul, capacité nulle et certificats de blocage. Une fois le stub complété, il retrace la courbe sur les six budgets et les 30 graines de la section 4.

In [8]:
def descente_2flip(x0, poids, valeurs, capacite, budget, evaluer):
    # TODO : evaluer(depuis, voisin) compte une evaluation et renvoie sa valeur
    # ou None si inadmissible. Retourner x, valeur, chemin, evaluations, raison.
    return None

def verifier_2flip(x0, poids, valeurs, capacite, budget):
    evenements = []
    def evaluer(depuis, voisin):
        assert len(evenements) < budget
        u, v = list(depuis), list(voisin)
        assert len(u) == len(v) == len(poids)
        assert all(b in (0, 1) for b in u + v) and admissible(u, poids, capacite)
        assert 1 <= sum(a != b for a, b in zip(u, v)) <= 2
        evenements.append((u, v))
        return valeur(v, valeurs) if admissible(v, poids, capacite) else None
    r = descente_2flip(list(x0), poids, valeurs, capacite, budget, evaluer)
    if r is None:
        return None
    assert set(r) == {'x', 'valeur', 'chemin', 'evaluations', 'raison'}
    chemin = r['chemin']
    assert chemin and chemin[0] == list(x0) and chemin[-1] == r['x']
    assert r['evaluations'] == len(evenements) <= budget
    assert r['raison'] in ('budget', 'blocage')
    assert all(len(x) == len(poids) and all(b in (0, 1) for b in x)
               and admissible(x, poids, capacite) for x in chemin)
    assert valeur(r['x'], valeurs) == r['valeur']
    debut = 0
    for u, v in zip(chemin, chemin[1:]):
        assert 1 <= sum(a != b for a, b in zip(u, v)) <= 2
        assert valeur(v, valeurs) > valeur(u, valeurs)
        indice = next((i for i in range(debut, len(evenements)) if evenements[i] == (u, v)), None)
        assert indice is not None
        debut = indice + 1
    if r['raison'] == 'blocage':
        # Un certificat requiert tous les voisins depuis l'etat final.
        x = r['x']
        vus = {tuple(v) for u, v in evenements[debut:] if u == x}
        attendus = set()
        for taille in (1, 2):
            for indices in itertools.combinations(range(len(x)), taille):
                y = list(x)
                for i in indices:
                    y[i] ^= 1
                attendus.add(tuple(y))
        assert attendus <= vus
        assert all(not admissible(list(y), poids, capacite) or valeur(list(y), valeurs) <= r['valeur'] for y in attendus)
    else:
        assert len(evenements) == budget
    return r

a = verifier_2flip([0], [1], [5], 1, 5)
b = verifier_2flip([1, 0], [1, 1], [1, 4], 1, 10)
c = verifier_2flip([0, 0], [1, 2], [3, 4], 0, 10)
z = verifier_2flip([0, 0], [2, 1], [1, 2], 1, 0)
if any(r is None for r in (a, b, c, z)):
    print("Exercice 3 a completer")
else:
    assert (a['valeur'], b['valeur'], c['valeur'], z['valeur']) == (5, 4, 0, 0)
    assert z['evaluations'] == 0 and z['raison'] == 'budget'
    print("Exercice 3 : traces observees, budgets et fixtures valides")
    for budget in (20, 50, 100, 200, 400, 800):
        gaps = []
        for seed in range(30):
            p, v, cap = instance_knapsack(24, 1000 + seed)
            rng = random.Random(2000 + seed)
            x = [rng.randint(0, 1) for _ in p]
            if not admissible(x, p, cap):
                x = [0] * len(p)
            r = verifier_2flip(x, p, v, cap, budget)
            gaps.append(1 - r['valeur'] / dp_exact(p, v, cap))
        print(f"1 ou 2 flips : budget {budget}, gap median {statistics.median(gaps):.1%}")


Exercice 3 a completer


## 5. Expérience ICT candidate, limites, exercices

**Expérience ICT candidate** : le protocole qualité-budget de §4, appliqué à une descente **multi-regards** (op 12) — budget partagé entre deux potentiels différents, courbe comparée au budget intégrairement alloué au regard le plus performant en isolation. Tester si le partage améliore ou non le meilleur regard isolé est une hypothèse expérimentale ; les observations de §3 ne tranchent pas cette comparaison.

**Limites** (bornes explicites) : (1) substrat sac à dos 0/1 seul — voisinage bit-flip unimodal par objet ; (2) hill-climbing déterministe — pas de recuit, pas de restarts (c'est l'exercice 1, pas le corps) ; (3) la comparaison MEALPy reste le territoire de Search-11 — ce notebook énonce la loi, il ne refait pas le tour d'horizon.

**Exercices** — les stubs s'exécutent sans erreur (politique du dépôt : compléter, pas débloquer).

## Conclusion

| Exigence de distillation | Où elle vit |
|---|---|
| Source primaire firsthand | `mimo_lean/Descent.lean` — `descent_flips_le_barrier` l.110, `descent_target_before_ceiling` l.144, thèse op 11 l.148 |
| Objet formel | `descente_sous_budget` : potentiel strictement décroissant, barrière LP, plafond d'évaluations |
| Claim exact, exercé | §2 : 60 exécutions, 0 violation `hstrict`, 0 violation `hbarrier`, 0 dépassement de plafond — *[exercée ici]* |
| Contre-claim, exhibé | §3 : `hnostall` fausse — blocages hors cible et arrêts sur budget mesurés dans les sorties |
| Expérience ICT candidate | §5 : qualité-budget d'une descente multi-regards (op 12) |

**Attestation** : 2ᵉ attestation de l'opération 11 « Descendre sous budget » (EPIC #12204), sur substrat indépendant de l'attestation Lean — empirique-notebook, stdout réels committés. La promotion en TABLE appartient à la relecture froide (A7).

## À retenir

Un balayage complet sans amélioration établit un optimum local, et non global. Un budget d'évaluations épuisé ne certifie pas la terminalité. Le coût naturel `floor(lp) - valeur` borne les flips acceptés.

## Vérifiez votre compréhension

<details><summary>Pourquoi un balayage partiel ne suffit-il pas ?</summary>Un voisin améliorant peut ne pas avoir été examiné.</details>

<details><summary>Quel compteur borne le résultat Lean ?</summary>Les flips acceptés, pas les évaluations de voisins.</details>

## Pour aller plus loin

Lire `mimo_lean/Descent.lean` pour les hypothèses formelles et Search-11 pour les métaheuristiques. Reprendre les trois exercices : redémarrages, barrière et voisinage élargi.


***

## Références et provenance

- EPIC #12204 (table des opérations) : deuxième attestation de l’opération 11 après `mimo_lean/Descent.lean` (première attestation Lean-formel).
